In [ ]:
# pip install fastapi uvicorn psycopg2-binary sqlalchemy faiss-cpu sentence-transformers

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv('MyData.csv', encoding='utf-8')


In [3]:
data

,ID,Date,Description,Priorité
0,INC00000522,2021-12-24,Transfert mail vers BMCI BACK OFFICE MONÉTIQUE,BAS
1,INC00000738,2019-08-08,Extraction des gratuites package suspendue,BAS
2,INC00000741,2024-10-03,Incident SAB : dossier non chargé dans Process...,BAS
3,INC00000661,2019-07-09,Rejets comptables sur export SAP,BAS
4,INC00000412,2023-02-23,Indisponibilité temporaire du système IRIS,MODÉRÉE
...,...,...,...,...
848,INC00000503,2020-12-05,Spool EICED001P1 non généré,BAS
849,INC00000048,2020-09-06,Spool EICED001P1 non généré,TRÈS BAS
850,INC00000033,2025-02-06,Transfert demande d'habilitation DA provisoire,TRÈS BAS
851,INC00000780,2023-05-11,Blocage au changement de classe de sécurité,BAS


In [4]:
import re

def nettoyer_texte(texte):
    """Nettoyage simple : suppression des caractères spéciaux, mise en minuscule"""
    texte = texte.lower()
    texte = re.sub(r'[^\w\s]', '', texte)
    texte = re.sub(r'\s+', ' ', texte).strip()
    return texte


In [5]:
data['description_clean'] = data['Description'].apply(nettoyer_texte)

In [6]:
data.head()

,ID,Date,Description,Priorité,description_clean
0,INC00000522,2021-12-24,Transfert mail vers BMCI BACK OFFICE MONÉTIQUE,BAS,transfert mail vers bmci back office monétique
1,INC00000738,2019-08-08,Extraction des gratuites package suspendue,BAS,extraction des gratuites package suspendue
2,INC00000741,2024-10-03,Incident SAB : dossier non chargé dans Process...,BAS,incident sab dossier non chargé dans process c...
3,INC00000661,2019-07-09,Rejets comptables sur export SAP,BAS,rejets comptables sur export sap
4,INC00000412,2023-02-23,Indisponibilité temporaire du système IRIS,MODÉRÉE,indisponibilité temporaire du système iris


In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Chargement du modèle BERT
modele_bert = SentenceTransformer("all-MiniLM-L6-v2")

def encoder_description(description):
    """Transforme une description en vecteur BERT"""
    description_propre = nettoyer_texte(description)
    vecteur = modele_bert.encode([description_propre])
    return vecteur[0].astype('float32')  # Format requis pour FAISS

In [8]:
# Encoder toutes les descriptions en une seule fois
vectors = modele_bert.encode(data['description_clean'].tolist())


In [9]:
import faiss

dimension = 384  # Taille des vecteurs BERT
index_faiss = faiss.IndexFlatL2(dimension)  

In [10]:
faiss_to_postgres = {}

In [11]:
# Ajout de tous les vecteurs dans FAISS
index_faiss.add(np.array(vectors, dtype='float32'))


# Vérifier le nombre d'éléments indexés
print(f"Nombre d'éléments indexés dans FAISS : {index_faiss.ntotal}")


Nombre d'éléments indexés dans FAISS : 853


In [12]:
query = "Problème de connexion à la base de données"
vec_query = encoder_description(query)
indices, distances = index_faiss.search(np.array([vec_query], dtype='float32'), k=5)

# Afficher les descriptions correspondantes
# session = SessionLocal()
# for idx in indices[0]:
#     incident = data.query(Incident).offset(idx).limit(1).first()
#     print(f"Incident #{incident.id}: {incident.description}")
# session.close()


#afficher descriptions correspondantes depuis data
for idx in indices[0]:
    idx = int(idx)  # Convertir l'index en entier
    print(f"Incident #{data.iloc[idx]['id_incident']}: {data.iloc[idx]['description']}")
    print(f"Distance: {distances[0][idx]}")
    print("-" * 40)


KeyError: 'id_incident'

In [ ]:
# pourquoi les resultats sont-ils les meme?
# Les résultats sont les mêmes car nous avons utilisé le même modèle BERT pour encoder la requête et les descriptions.
# Les vecteurs encodés sont comparés dans l'index FAISS, qui est basé sur la distance euclidienne.
# Pour obtenir des résultats différents, il faudrait utiliser un modèle différent ou modifier la requête.
# utiliser un modèle différent ou modifier la requête pour obtenir des résultats différents.
# Pour utiliser un modèle différent, vous pouvez essayer d'autres modèles disponibles dans la bibliothèque `sentence-transformers`.
# Par exemple, vous pouvez remplacer "all-MiniLM-L6-v2" par un autre modèle comme "distilbert-base-nli-stsb-mean-tokens".
# Pour modifier la requête, vous pouvez changer le texte de la requête pour qu'il corresponde à un autre incident ou à une autre description.


In [ ]:
print(encoder_description("Ceci est un exemple de description."))

[-1.61952041e-02  6.17311411e-02  7.51508772e-03 -2.20188834e-02
 -1.99037092e-03  5.30362176e-03  9.10033733e-02  7.18366504e-02
  6.66384995e-02 -3.02938893e-02  3.50321531e-02 -1.66120321e-01
  3.84936780e-02 -4.80977632e-03 -1.33561166e-02 -1.07164592e-01
  4.68625799e-02  3.64167392e-02 -3.18408594e-03  4.66006882e-02
  6.65993094e-02 -2.44973842e-02 -2.09199376e-02  9.14924145e-02
 -7.25407749e-02  1.15802353e-02 -8.69287178e-04  2.97512170e-02
 -1.80446599e-02 -8.17423239e-02  7.07307756e-02  4.03873101e-02
  7.96118826e-02 -2.89324522e-02  6.59964606e-02 -5.47440071e-03
 -1.39213940e-02 -3.51571813e-02  4.22835313e-02 -6.25755936e-02
 -1.10586479e-01 -2.70376317e-02 -6.37628585e-02 -2.63885614e-02
  3.84081937e-02  8.12099595e-03  1.03284689e-02  1.86081696e-02
 -5.93955368e-02 -2.21525729e-02 -5.05293831e-02 -2.13018954e-02
  1.02079064e-02 -1.71145890e-02  1.63144488e-02  2.09495449e-03
 -6.13982975e-03 -6.21841587e-02  5.73591590e-02  2.70976732e-03
  4.71644215e-02 -7.37719

In [13]:
import faiss

dimension = 384  # Taille des vecteurs BERT
index_faiss = faiss.IndexFlatL2(dimension)  

# Ajout des vecteurs à l'index
def ajouter_a_faiss(vecteur):
    index_faiss.add(np.array([vecteur], dtype='float32'))

# Recherche dans FAISS
def rechercher_similaires_faiss(vecteur_query, top_k=5):
    distances, indices = index_faiss.search(np.array([vecteur_query], dtype='float32'), top_k)
    return indices[0], distances[0]


In [19]:
ajouter_a_faiss(encoder_description("Ceci est un exemple de description."))
ajouter_a_faiss(encoder_description("Un autre exemple de description."))

In [14]:
print(rechercher_similaires_faiss(encoder_description("cette description est mieux"), top_k=1))

(array([-1], dtype=int64), array([3.4028235e+38], dtype=float32))


In [17]:
# taille de faiss
print("Taille de l'index FAISS :", index_faiss.ntotal)

Taille de l'index FAISS : 2


In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
import faiss
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer
import spacy
from collections import Counter
import pickle
import json
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from transformers import AutoTokenizer, AutoModel
import torch
from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import PCA

In [16]:
try:
    stop_words = set(stopwords.words('french'))
except:
    nltk.download('stopwords')
    nltk.download('punkt')
    stop_words = set(stopwords.words('french'))

In [17]:
def __init__(self, language='french'):
        self.language = language
        self.models = {}
        self.vectorizers = {}
        self.similarity_matrices = {}
        self.incident_embeddings = {}
        self.clustered_data = {}

        self._initialize_models()

In [18]:
def _initialize_models(self):
        """Initialise tous les modèles de similarité"""
        print(" Initialisation des modèles avancés...")
        
        # 1. Modèle TF-IDF optimisé
        self.vectorizers['tfidf'] = TfidfVectorizer(
            max_features=10000,
            ngram_range=(1, 3),  # Unigrams, bigrams, trigrams
            stop_words=list(stop_words),
            min_df=2,
            max_df=0.85,
            sublinear_tf=True,  # Améliore les performances
            use_idf=True
        )
        
        # 2. Modèle Sentence-BERT multilingue
        self.models['sbert'] = SentenceTransformer('distiluse-base-multilingual-cased')
        
        self.models['bert'] = SentenceTransformer('all-MiniLM-L6-v2')  # Modèle plus léger et rapide


        
        # 3. Modèle CamemBERT pour le français (si disponible)
        try:
            self.models['camembert_tokenizer'] = AutoTokenizer.from_pretrained('camembert-base')
            self.models['camembert_model'] = AutoModel.from_pretrained('camembert-base')
            print("✅ CamemBERT chargé avec succès")
        except:
            print("⚠️ CamemBERT non disponible, utilisation de SBERT uniquement")
        
        # 4. Stemmer français
        self.stemmer = SnowballStemmer('french')
        
        # 5. Index FAISS pour recherche ultra-rapide
        self.faiss_indices = {}
        

In [19]:
def preprocess_text(self, text):
        """Préprocessing avancé du texte"""
        if pd.isna(text) or text == '':
            return ""
        
        # Conversion en minuscules
        text = str(text).lower()
        
        # Suppression des caractères spéciaux mais conservation des accents
        text = re.sub(r'[^\w\sàâäçéèêëïîôöùûüÿ-]', ' ', text)
        
        # Suppression des espaces multiples
        text = re.sub(r'\s+', ' ', text)
        
        # Tokenisation et suppression des mots vides
        words = word_tokenize(text, language='french')
        words = [self.stemmer.stem(word) for word in words 
                if word not in stop_words and len(word) > 2]
        
        return ' '.join(words)

In [20]:
def extract_technical_features(self, text):
    """
    Extraction des caractéristiques techniques d'un ticket d'incident IT, adaptée au contexte APS BDSI/BMCI.
    Permet d'enrichir la description d'un ticket avec des métadonnées pertinentes pour l'analyse de similarité.
    """
    features = {}

    # 🔍 Détection des services et applications courantes dans l'environnement bancaire/APS
    services = [
        'SAP', 'Cliker', 'MySQL', 'PostgreSQL', 'Oracle', 'Exchange', 'Active Directory', 
        'Kubernetes', 'Docker', 'Jenkins', 'Tomcat', 'IIS', 'Apache', 'Nginx', 
        'CFT', 'Autosys', 'DataLake', 'SFTP', 'Kafka', 'ElasticSearch'
    ]
    found_services = [s for s in services if s.lower() in text.lower()]
    features['services_detected'] = found_services



    import re
    

    # 📊 Détection de mots-clés liés à l'infrastructure et aux incidents bancaires
    infra_keywords = ['ticket', 'incident', 'crash', 'latence', 'indisponibilité', 'bascule', 'redémarrage', 'backup', 'plan batch', 'job', 'JIL', 'KSH', 'SAP']
    features['infra_keywords'] = [kw for kw in infra_keywords if kw.lower() in text.lower()]

    # 📅 Détection des dates et heures
    features['dates_times'] = re.findall(r'\b\d{2}/\d{2}/\d{4}\b|\b\d{4}-\d{2}-\d{2}\b|\b\d{2}:\d{2}(:\d{2})?\b', text)

    # 📂 Détection des fichiers log, scripts et fichiers de configuration (KSH, JIL)
    files = re.findall(r'\b[\w/\\.-]+\.(log|txt|csv|conf|cfg|sh|ksh|jil|xml|json)\b', text)
    features['files_detected'] = files

    # 📈 Détection des modules métiers ou processus bancaires (APS context)
    modules = ['Module Client', 'Module Comptes', 'Datalake', 'Flux CFT', 'KSH Script', 'Job Autosys', 'Batch', 'Reporting']
    found_modules = [m for m in modules if m.lower() in text.lower()]
    features['modules'] = found_modules

    return features


In [21]:
def fit_transform_tfidf(self, incidents_df):
        """Entraînement et transformation TF-IDF"""
        print("📊 Entraînement du modèle TF-IDF...")
        
        processed_texts = incidents_df['description'].apply(self.preprocess_text)
        
        tfidf_matrix = self.vectorizers['tfidf'].fit_transform(processed_texts)
        
        similarity_matrix = cosine_similarity(tfidf_matrix)
        
        self.incident_embeddings['tfidf'] = tfidf_matrix
        self.similarity_matrices['tfidf'] = similarity_matrix
        
        print(f"✅ TF-IDF: {tfidf_matrix.shape[0]} incidents, {tfidf_matrix.shape[1]} features")
        return tfidf_matrix

In [22]:
def find_similar_incidents(self, query_idx, method='hybrid', top_k=10, threshold=0.3):
        """Recherche des incidents similaires"""
        if method not in self.similarity_matrices:
            raise ValueError(f"Méthode {method} non disponible")
        
        similarities = self.similarity_matrices[method][query_idx]
      
        similar_indices = np.argsort(similarities)[::-1]
        
        
        filtered_indices = []
        filtered_scores = []
        
        for idx in similar_indices:
            if idx != query_idx and similarities[idx] > threshold:
                filtered_indices.append(idx)
                filtered_scores.append(similarities[idx])
                
                if len(filtered_indices) >= top_k:
                    break
        
        return filtered_indices, filtered_scores

In [23]:
incidents_df = pd.read_csv('MyData.csv', encoding='utf-8')

fit_transform_tfidf(incidents_df)

print(find_similar_incidents(0, method='tfidf', top_k=5, threshold=0.2))

TypeError: fit_transform_tfidf() missing 1 required positional argument: 'incidents_df'

In [24]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


incidents_df = pd.read_csv('Mydata.csv', encoding='utf-8')


incidents_df.head()


,ID,Date,Description,Priorité
0,INC00000522,2021-12-24,Transfert mail vers BMCI BACK OFFICE MONÉTIQUE,BAS
1,INC00000738,2019-08-08,Extraction des gratuites package suspendue,BAS
2,INC00000741,2024-10-03,Incident SAB : dossier non chargé dans Process...,BAS
3,INC00000661,2019-07-09,Rejets comptables sur export SAP,BAS
4,INC00000412,2023-02-23,Indisponibilité temporaire du système IRIS,MODÉRÉE


In [26]:
try:
    stop_words = set(stopwords.words('french'))
except:
    nltk.download('stopwords')
    nltk.download('punkt')
    stop_words = set(stopwords.words('french'))

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 3),
    stop_words=list(stop_words),
    min_df=2,
    max_df=0.85,
    sublinear_tf=True
)

# Transformation des descriptions
processed_texts = incidents_df['Description'].fillna('').astype(str)
tfidf_matrix = vectorizer.fit_transform(processed_texts)

# Calcul des similarités
similarity_matrix = cosine_similarity(tfidf_matrix)

print(f"✅ TF-IDF entraîné sur {tfidf_matrix.shape[0]} incidents, {tfidf_matrix.shape[1]} features.")

✅ TF-IDF entraîné sur 853 incidents, 295 features.


In [28]:
def find_similar_incidents(query_idx, top_k=5, threshold=0.2):
    similarities = similarity_matrix[query_idx]
    similar_indices = similarities.argsort()[::-1]
    
    results = []
    for idx in similar_indices:
        if idx != query_idx and similarities[idx] >= threshold:
            results.append((idx, similarities[idx]))
            if len(results) >= top_k:
                break
    return results

# Exécution
similar_incidents = find_similar_incidents(0, top_k=5, threshold=0.2)

# Affichage des résultats
for idx, score in similar_incidents:
    print(f"Incident similaire: {incidents_df.iloc[idx]['Description']}")
    print(f"Score: {score:.3f}")
    print('-' * 40)


Incident similaire: Transfert mail vers BMCI BACK OFFICE MONÉTIQUE
Score: 1.000
----------------------------------------
Incident similaire: Transfert mail vers BMCI BACK OFFICE MONÉTIQUE
Score: 1.000
----------------------------------------
Incident similaire: Transfert mail vers BMCI BACK OFFICE MONÉTIQUE
Score: 1.000
----------------------------------------
Incident similaire: Transfert mail vers BMCI BACK OFFICE MONÉTIQUE
Score: 1.000
----------------------------------------
Incident similaire: Transfert mail vers BMCI BACK OFFICE MONÉTIQUE
Score: 1.000
----------------------------------------


In [29]:
def search_similar_incidents(query_text, top_k=5, threshold=0.2):
    # Vectoriser la requête
    query_vector = vectorizer.transform([query_text])
    
    # Calculer les similarités
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    # Trier les indices selon la similarité
    similar_indices = similarities.argsort()[::-1]
    
    # Filtrer selon le seuil
    results = []
    for idx in similar_indices:
        if similarities[idx] >= threshold:
            results.append((idx, similarities[idx]))
            if len(results) >= top_k:
                break
    
    return results


In [30]:
bert_model = SentenceTransformer('distiluse-base-multilingual-cased')

incident_embeddings = {}
similarity_matrices = {}
faiss_indices = {}

In [38]:


def fit_transform_bert( incidents_df):
        
        """Entraînement et transformation Sentence-BERT"""
        print("🧠 Génération des embeddings Sentence-BERT...")
        
        descriptions = incidents_df['Description'].fillna('').tolist()

        # 2️⃣ Génération des embeddings avec Sentence-BERT
        embeddings = bert_model.encode(
                descriptions,
                batch_size=32,
                show_progress_bar=True,
                normalize_embeddings=True  # Normalisation pour FAISS
        )

        # 3️⃣ Calcul de la matrice de similarité
        similarity_matrix = cosine_similarity(embeddings)

        # 4️⃣ Stockage des résultats
        incident_embeddings['bert'] = embeddings
        similarity_matrices['bert'] = similarity_matrix

        # 5️⃣ Création et remplissage de l'index FAISS
        dimension = embeddings.shape[1]
        index = faiss.IndexFlatIP(dimension)
        index.add(embeddings.astype('float32'))

        faiss_indices['bert'] = index

        print(f"✅ BERT: {embeddings.shape[0]} incidents, {embeddings.shape[1]}D embeddings")
        return embeddings

In [32]:

technical_weights = {
    'cft': 0.4,
    'autosys': 0.4,
    'ksh': 0.4,
    'batch': 0.3,
    'job': 0.2,
    'datalake': 0.3,
    'sap': 0.0,
    'cliker': 0.4,
    'compte': 0.3,
    'swift': 0.5,
    'virement': 0.4,
    'fixing': 0.4,
    'paiement': 0.3,
    'bloqué': 0.4,
    'carte': 0.4,
    'crm': 0.3,
    'fixing': 0.3,
    'oracle': 0.2,
    'incident': 0.0, 
    'serveur': 0.0,
    'client': 0.0
}


In [ ]:
# Appelle la fonction sur ton DataFrame
# fit_transform_bert(incidents_df)

# # Recherche d'incidents similaires
# query = "Erreur sur le job Autosys dans le Datalake"
# query_embedding = bert_model.encode([query], normalize_embeddings=True).astype('float32')

# scores, indices = faiss_indices['bert'].search(query_embedding, top_k=5)

# # Affiche les résultats
# for i, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
#     print(f"{i}. {incidents_df.iloc[idx]['description']} | Score : {score:.3f}")


In [33]:
def compute_technical_score(text):
    score = 0.0
    text_lower = text.lower()
    for term, weight in technical_weights.items():
        if term in text_lower:
            score += weight
    return score


In [43]:
def search_similar_incidents(query_text, top_k=5, threshold=0.2):
    # Vectoriser la requête
    query_vector = vectorizer.transform([query_text])
    
    # Calculer les similarités TF-IDF
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    # Ajouter un boost basé sur les termes techniques
    query_score = compute_technical_score(query_text)
    
    boosted_similarities = []
    for idx, sim in enumerate(similarities):
        boost = compute_technical_score(incidents_df.iloc[idx]['Description'])
        final_score = sim + 0.2 * (query_score + boost)  # Pondération hybride
        boosted_similarities.append((idx, final_score))
    
    # Trier et filtrer selon le seuil
    boosted_similarities = sorted(boosted_similarities, key=lambda x: x[1], reverse=True)
    
    results = []
    for idx, score in boosted_similarities:
        if score >= threshold:
            results.append((idx, score))
            if len(results) >= top_k:
                break
    return results


In [40]:
def search_similar_incidents(query_text, top_k=5, threshold=0.2):
    """
    Recherche des incidents similaires à une requête texte en utilisant BERT + FAISS + boost technique.
    """
    # 1️⃣ Vectoriser la requête avec BERT
    query_embedding = bert_model.encode([query_text], normalize_embeddings=True).astype('float32')

    # 2️⃣ Recherche des similarités via FAISS
    scores, indices = faiss_indices['bert'].search(query_embedding, top_k=100)  # On prend plus large pour filtrer

    # 3️⃣ Calcul du boost technique sur la requête
    query_score = compute_technical_score(query_text)

    # 4️⃣ Appliquer le boost sur chaque résultat
    boosted_similarities = []
    for idx, raw_score in zip(indices[0], scores[0]):
        desc = incidents_df.iloc[idx]['description']
        boost = compute_technical_score(desc)
        final_score = raw_score + 0.2 * (query_score + boost)
        boosted_similarities.append((idx, final_score))

    # 5️⃣ Trier et filtrer selon le seuil
    boosted_similarities = sorted(boosted_similarities, key=lambda x: x[1], reverse=True)

    results = []
    for idx, score in boosted_similarities:
        if score >= threshold:
            results.append((idx, score))
            if len(results) >= top_k:
                break

    return results


In [44]:
query = "Erreur sur le job Autosys dans le Datalake"
fit_transform_bert(pd.read_csv('MyData.csv', encoding='utf-8'))
results = search_similar_incidents(query, top_k=5, threshold=0.2)

for i, (idx, score) in enumerate(results, 1):
    print(f"{i}. {incidents_df.iloc[idx]['Description']} | Score final : {score:.3f}")


🧠 Génération des embeddings Sentence-BERT...


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

✅ BERT: 853 incidents, 512D embeddings
1. Facturation bloquée suite à une erreur sur INSTANT PAYMENT | Score final : 0.455
2. Facturation bloquée suite à une erreur sur INSTANT PAYMENT | Score final : 0.455
3. Facturation bloquée suite à une erreur sur INSTANT PAYMENT | Score final : 0.455
4. Facturation bloquée suite à une erreur sur INSTANT PAYMENT | Score final : 0.455
5. Facturation bloquée suite à une erreur sur INSTANT PAYMENT | Score final : 0.455


In [55]:
query_text = "Erreur sur le job Autosys dans le Datalake"
results = search_similar_incidents(query_text, top_k=5, threshold=0.3)

for idx, score in results:
    print(f"Incident : {incidents_df.iloc[idx]['description']}")
    print(f"Score : {score:.3f}")
    print('-' * 50)


AssertionError: 

In [27]:
def search_similar_incidents(query_text, top_k=5, threshold=0.2):
    print(f"🔎 Requête : {query_text}")
    
    # 1️⃣ Vectoriser la requête
    query_vector = vectorizer.transform([query_text])
    print("✅ Vecteur TF-IDF de la requête généré.")
    
    # 2️⃣ Calcul des similarités brutes
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    print(f"📈 Similarités brutes (avant pondération) :")
    print(similarities)
    
    # 3️⃣ Calcul du score technique (bonus)
    query_score = compute_technical_score(query_text)
    print(f"🚀 Score pondéré de la requête (mots techniques) : {query_score:.3f}")
    
    boosted_similarities = []
    for idx, sim in enumerate(similarities):
        boost = compute_technical_score(incidents_df.iloc[idx]['description'])
        final_score = sim + 0.2 * (query_score + boost)
        boosted_similarities.append((idx, final_score))
        print(f"Incident {idx} | TF-IDF : {sim:.3f} | Bonus : {boost:.3f} | Score final : {final_score:.3f}")
    
    # 4️⃣ Tri et filtrage
    boosted_similarities = sorted(boosted_similarities, key=lambda x: x[1], reverse=True)
    
    results = []
    for idx, score in boosted_similarities:
        if score >= threshold:
            results.append((idx, score))
            if len(results) >= top_k:
                break
    
    print("\n🎯 Résultats finaux :")
    for i, (idx, score) in enumerate(results, 1):
        print(f"{i}. Incident {idx} | Score : {score:.3f}")
    
    return results


In [ ]:
# Exemple d'incident
query_text = "Erreur sur le module client, serveur indisponible"

# Rechercher les incidents similaires
similar_incidents = search_similar_incidents(query_text, top_k=5, threshold=0.2)

# Affichage des résultats
for idx, score in similar_incidents:
    print(f"Incident similaire : {incidents_df.iloc[idx]['description']}")
    print(f"Score : {score:.3f}")
    print('-' * 40)

Incident similaire : Un seul utilisateur rencontrent une situation de erreur mineure sur le logiciel GPI.
Score : 0.269
----------------------------------------
Incident similaire : Une agence régionale rencontrent une situation de erreur mineure sur le logiciel CLICKER.
Score : 0.268
----------------------------------------
Incident similaire : Une agence régionale rencontrent une situation de erreur mineure sur le logiciel SBS.
Score : 0.265
----------------------------------------
Incident similaire : Un seul utilisateur rencontrent une situation de erreur mineure sur le logiciel DOCUBASE.
Score : 0.250
----------------------------------------
Incident similaire : Un seul utilisateur rencontrent une situation de erreur mineure sur le logiciel DOCUBASE.
Score : 0.250
----------------------------------------


In [28]:
query_text = "Erreur critique sur le serveur Autosys, port 8080, CFT en échec"
results = search_similar_incidents(query_text, top_k=5, threshold=0.3)


🔎 Requête : Erreur critique sur le serveur Autosys, port 8080, CFT en échec
✅ Vecteur TF-IDF de la requête généré.
📈 Similarités brutes (avant pondération) :
[0.08595726 0.         0.         0.         0.         0.
 0.20635066 0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.08565698 0.10281608 0.         0.         0.         0.08314123
 0.13179444 0.08538828 0.08574076 0.         0.13926148 0.
 0.12921819 0.06873613 0.17441466 0.         0.         0.07650921
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.07756528 0.         0.
 0.08601871 0.         0.08576569 0.         0.         0.
 0.         0.12916183 0.         0.         0.         0.
 0.24026917 0.06842883 0.23952779 0.         0.18486079 0.
 0.06405995 0.06713448 0.         0.         0.         0.
 0.         0.         0.1206964  0.24437774 0.         0.
 0.         0.         0.         0.06934047 0.         0.0